# 04 — Fases de entrenamiento MoE (consigna §7 + hardware §8)

Este notebook **no sustituye** el entrenamiento pesado en GPU: define **el calendario**, la **pérdida auxiliar Switch** y enlaza con el **notebook 03** (router + CLS).

**Orden recomendado (profesor / consigna):**
1. **Método B — Fase 1:** entrenar cada experto en su dataset (checkpoints por experto).
2. **Extracción CLS + ablation** (notebook `03_Pipeline_Router_MoE.ipynb`).
3. **Método B — Fase 2:** entrenar solo el **router ViT+Linear** con expertos **congelados**, DataLoader mixto proporcional.
4. **Método B — Fase 3:** fine-tuning conjunto (LR más bajo en expertos que en router) + early stopping.

**Método A (alternativa):** un solo script 40 épocas con tres tramos (router primero, luego últimas capas 2D, luego todo).


In [ ]:
import sys
from pathlib import Path

# Raíz del proyecto (padre de /notebooks)
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn.functional as F

from moe_training_phases import (
    METHOD_A_PHASES,
    METHOD_B_PHASES,
    get_method_a_phase_for_epoch,
    switch_transformer_auxiliary_loss,
    total_loss_switch,
    describe_schedule_markdown,
    freeze_module,
    timm_unfreeze_last_n_blocks,
)

print('Proyecto:', ROOT)
print('Imports OK.')


## 1. Tabla de fases (consigna)

| Método | Tramo | Épocas | Qué entrena | LR típico |
|--------|--------|--------|-------------|-----------|
| **A** | 1 | 1–10 | Router + cabezas; expertos congelados | 1e-3 |
| **A** | 2 | 11–25 | Últimas capas expertos **2D**; router | 1e-4 |
| **A** | 3 | 26–40 | Fine-tuning global (3D con checkpointing) | 1e-5 |
| **B** | 1 | — | Solo expertos en cada dataset | — |
| **B** | 2 | ~30 | Solo router (expertos congelados), loader mixto | router 1e-3 |
| **B** | 3 | ~30 | Todo junto | expertos 1e-5, router 1e-4 |

**Pérdida total:** `L = L_task + α · L_aux` con `α ∈ [0.01, 0.1]` (balanceo carga; penalización si `max(f_i)/min(f_i) > 1.30`).


In [ ]:
from IPython.display import Markdown
display(Markdown(describe_schedule_markdown()))


## 2. Pérdida auxiliar Switch (verificación numérica)

`router_probs` debe ser **softmax** sobre los 5 expertos `[B, 5]`.


In [ ]:
B, E = 8, 5
logits = torch.randn(B, E)
probs = F.softmax(logits, dim=1)

aux = switch_transformer_auxiliary_loss(probs, num_experts=E)
task = torch.tensor(0.5)  # ejemplo L_task
alpha = 0.05
total = total_loss_switch(task, probs, E, alpha)

print('L_aux:', aux.item())
print('L_total:', total.item())


## 3. Método A: qué hacer en cada época

Usa `get_method_a_phase_for_epoch(epoch)` para leer LR y política de congelación. **Congelar/descongelar** expertos concreto depende de cómo instancies los 5 modelos (timm); la función `timm_unfreeze_last_n_blocks` es una plantilla para expertos que tengan `.blocks`.


In [ ]:
for ep in [1, 10, 11, 25, 26, 40]:
    ph = get_method_a_phase_for_epoch(ep)
    print(f'Época {ep}: {ph.name} | lr_router={ph.lr_router}')


## 4. Hardware obligatorio (consigna §8)

- **FP16:** `torch.amp.autocast('cuda')` + `GradScaler`.
- **Gradient accumulation:** p. ej. 4 pasos → batch efectivo 32.
- **Gradient checkpointing:** expertos 3D (LUNA, páncreas).
- **DataParallel:** `nn.DataParallel(model)` si hay 2 GPUs.

En el bucle real: un paso hacia adelante con **top-1** router → solo el experto elegido recibe backward (o implementación equivalente según tu código).


## 5. Esqueleto de bucle (Fase 2 Método B — router + expertos congelados)

Conecta aquí tu `VisionRouter` del notebook 03, los **cinco expertos cargados** desde checkpoints de la Fase 1, y el `DataLoader` mixto con `mixed_collate_fn`.

```text
# Pseudocódigo:
# for epoch in range(epochs_fase2):
#     for tensors, expert_id in loader:  # expert_id = supervisión débil por dataset
#         logits_r, cls = router(tensors)
#         probs = softmax(logits_r)
#         top_k = 1  → índice experto
#         logits_e = experts[k](tensors[k])  # solo una rama activa por muestra
#         L_task = CrossEntropy(logits_e, y_task)  # o BCE/Focal según dataset
#         L = L_task + alpha * L_aux(probs)
#         L.backward()
```

La **Routing Accuracy** del ablation se compara contra `expert_id` (etiqueta de dataset de origen).


In [ ]:
# Hiperparámetros sugeridos (consigna)
ACCUMULATION_STEPS = 4
ALPHA_AUX = 0.05  # calibrar en [0.01, 0.1]
EPOCHS_FASE2_METHOD_B = METHOD_B_PHASES[1].epochs

print('ACCUMULATION_STEPS:', ACCUMULATION_STEPS)
print('ALPHA_AUX:', ALPHA_AUX)
print('Épocas Fase 2 (Método B):', EPOCHS_FASE2_METHOD_B)
